<a href="https://colab.research.google.com/github/rishabhjune82/AI-attendance-System/blob/main/project_py_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# INSTALL REQUIRED LIBRARIES
!pip install streamlit opencv-python face-recognition numpy pandas pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 8.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 120.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 70.1 MB/s eta 0:00:00
  Created wheel for face-recognition-models: filename=face_recognition_models-0.3.0-py2.py3-none-any.whl size=100566166 sha256=e2e5b03704b1d484a8b04e3879e1809afc6c36a8fdb60cfc1f6ddf7ff3bf9684
  Stored in directory: /root/.cache/pip/wheels/8f/47/c8/f44c5aebb7507f7c8a2c0bd23151d732d0f0bd6884ad4ac635
Successfully built face-recognition-models


In [2]:
%%writefile app.py

import streamlit as st
import cv2
import face_recognition
import numpy as np
import pandas as pd
import os

from PIL import Image
from datetime import datetime


# ==========================================
# 1. CREATE FOLDERS
# ==========================================

if not os.path.exists("students"):
    os.makedirs("students")

if not os.path.exists("attendance"):
    os.makedirs("attendance")


# ==========================================
# 2. ATTENDANCE FILE
# ==========================================

ATTENDANCE_FILE = "attendance/attendance.csv"


if not os.path.exists(ATTENDANCE_FILE):

    df = pd.DataFrame(
        columns=[
            "Name",
            "Date",
            "Time",
            "Status"
        ]
    )

    df.to_csv(
        ATTENDANCE_FILE,
        index=False
    )
# FACE LOADING FUNCTION

def load_known_faces():

    known_encodings = []
    known_names = []

    # Check whether students folder exists
    if not os.path.exists("students"):
        return known_encodings, known_names

    # Read all student photos
    for file_name in os.listdir("students"):

        # Check image files
        if file_name.lower().endswith(
            (".jpg", ".jpeg", ".png")
        ):

            # Create complete image path
            image_path = os.path.join(
                "students",
                file_name
            )

            # Load student image
            image = face_recognition.load_image_file(
                image_path
            )

            # Detect face
            face_locations = face_recognition.face_locations(
                image
            )

            # We need exactly one face
            if len(face_locations) != 1:
                continue

            # Generate face encoding
            face_encodings = face_recognition.face_encodings(
                image,
                face_locations
            )

            if len(face_encodings) == 0:
                continue

            # Get encoding
            encoding = face_encodings[0]

            # Get student name from filename
            name = os.path.splitext(
                file_name
            )[0]

            # Store encoding
            known_encodings.append(
                encoding
            )

            # Store name
            known_names.append(
                name
            )

    return known_encodings, known_names
# FACE RECOGNITION FUNCTION

def recognize_face(image):

    # Load registered faces
    known_encodings, known_names = (
        load_known_faces()
    )

    # Check registered students
    if len(known_encodings) == 0:

        return "NO_REGISTERED_STUDENTS"

    # Detect faces in camera image
    face_locations = face_recognition.face_locations(
        image
    )

    # Check whether face exists
    if len(face_locations) == 0:

        return "NO_FACE"

    # Generate encoding for camera face
    face_encodings = face_recognition.face_encodings(
        image,
        face_locations
    )

    recognized_names = []

    # Compare every detected face
    for face_encoding in face_encodings:

        # Compare with registered faces
        distances = face_recognition.face_distance(
            known_encodings,
            face_encoding
        )

        # Find closest face
        best_match_index = np.argmin(
            distances
        )

        # Get distance
        best_distance = distances[
            best_match_index
        ]

        # Recognition threshold
        if best_distance < 0.50:

            name = known_names[
                best_match_index
            ]

        else:

            name = "Unknown"

        recognized_names.append(
            name
        )

    return recognized_names

    # ==========================================
# DELETE ATTENDANCE OLDER THAN 24 HOURS
# ==========================================

def delete_old_attendance():

    # Check if attendance file exists
    if not os.path.exists(ATTENDANCE_FILE):
        return

    # Read attendance file
    df = pd.read_csv(
        ATTENDANCE_FILE
    )

    # If there are no records, stop
    if df.empty:
        return

    # Combine Date and Time
    df["DateTime"] = pd.to_datetime(
        df["Date"] + " " + df["Time"],
        errors="coerce"
    )

    # Get current date and time
    current_time = datetime.now()

    # Calculate time 24 hours ago
    twenty_four_hours_ago = (
        current_time - pd.Timedelta(hours=24)
    )

    # Keep only records from the last 24 hours
    df = df[
        df["DateTime"] >= twenty_four_hours_ago
    ]

    # Remove temporary DateTime column
    df = df.drop(
        columns=["DateTime"]
    )

    # Save updated attendance
    df.to_csv(
        ATTENDANCE_FILE,
        index=False
    )

# ==========================================
# MARK ATTENDANCE
# ==========================================

def mark_attendance(name):

    # Read existing attendance
    df = pd.read_csv(
        ATTENDANCE_FILE
    )

    # Get today's date
    today = datetime.now().strftime(
        "%Y-%m-%d"
    )

    # Get current time
    current_time = datetime.now().strftime(
        "%H:%M:%S"
    )

    # Check whether student is already present
    already_present = df[
        (df["Name"] == name) &
        (df["Date"] == today)
    ]

    # Prevent duplicate attendance
    if not already_present.empty:

        return False, (
            f"{name} is already marked "
            "present today."
        )

    # Create new attendance record
    new_record = pd.DataFrame({

        "Name": [name],

        "Date": [today],

        "Time": [current_time],

        "Status": ["Present"]

    })

    # Add record
    df = pd.concat(
        [
            df,
            new_record
        ],
        ignore_index=True
    )

    # Save file
    df.to_csv(
        ATTENDANCE_FILE,
        index=False
    )

    return True, (
        f"Attendance marked for {name}."
    )


# ==========================================
# 3. STREAMLIT PAGE SETTINGS
# ==========================================

st.set_page_config(
    page_title="AI Smart Attendance",
    page_icon="🎓",
    layout="wide"
)


# ==========================================
# 4. TITLE
# ==========================================

st.title(
    "🎓 AI-Based Smart Attendance Monitoring System"
)

st.write(
    "Face Recognition Based Attendance System"
)


# ==========================================
# 5. SIDEBAR
# ==========================================

st.sidebar.title("Navigation")


menu = st.sidebar.selectbox(
    "Select Option",
    [
        "Home",
        "Register Student",
        "Take Attendance",
        "Attendance Records"
    ]
)


# ==========================================
# 6. HOME
# ==========================================

if menu == "Home":

    st.header("🏠 Home")

    st.write(
        """
        Welcome to the AI-Based Smart Attendance
        Monitoring System.
        """
    )

    st.success(
        "Application is running successfully!"
    )


# ==========================================
# 7. REGISTER STUDENT
# ==========================================

elif menu == "Register Student":

    st.header("👨‍🎓 Register Student")

    student_name = st.text_input(
        "Enter Student Name"
    )

    uploaded_file = st.file_uploader(
        "Upload Student Photo",
        type=["jpg", "jpeg", "png"]
    )

    if uploaded_file is not None:

        st.image(
            uploaded_file,
            caption="Student Photo",
            width=300
        )

    if st.button("Register Student"):

        if student_name.strip() == "":

            st.error(
                "Please enter student name."
            )

        elif uploaded_file is None:

            st.error(
                "Please upload a photo."
            )

        else:

            # Read image
            image = face_recognition.load_image_file(
                uploaded_file
            )

            # Detect faces
            face_locations = (
                face_recognition.face_locations(
                    image
                )
            )

            # No face
            if len(face_locations) == 0:

                st.error(
                    "No face detected."
                )

            # Multiple faces
            elif len(face_locations) > 1:

                st.error(
                    "Multiple faces detected. "
                    "Please upload a photo containing "
                    "only one person."
                )

            # Exactly one face
            else:

                file_path = os.path.join(
                    "students",
                    student_name + ".jpg"
                )

                with open(
                    file_path,
                    "wb"
                ) as file:

                    file.write(
                        uploaded_file.getbuffer()
                    )

                st.success(
                    f"{student_name} registered successfully!"
                )
# ==========================================
# TAKE ATTENDANCE
# ==========================================

elif menu == "Take Attendance":

    st.header("📸 Take Attendance")

    st.write(
        "Look at the camera and take a photo."
    )

    # Open camera
    camera_image = st.camera_input(
        "Take a photo"
    )

    # Check if photo was captured
    if camera_image is not None:

        # Load captured image
        image = face_recognition.load_image_file(
            camera_image
        )

        # Recognize face
        result = recognize_face(
            image
        )

        # ----------------------------------
        # NO REGISTERED STUDENTS
        # ----------------------------------

        if result == "NO_REGISTERED_STUDENTS":

            st.error(
                "❌ No registered students found."
            )

        # ----------------------------------
        # NO FACE
        # ----------------------------------

        elif result == "NO_FACE":

            st.error(
                "❌ No face detected."
            )

        # ----------------------------------
        # FACE RECOGNIZED
        # ----------------------------------

        else:

            for name in result:

                if name == "Unknown":

                    st.error(
                        "❌ Face not recognized."
                    )

                else:

                    st.success(
                        f"✅ Face recognized: {name}"
                    )

                    # Mark attendance
                    success, message = mark_attendance(
                        name
                    )

                    if success:

                        st.success(
                            message
                        )

                    else:

                        st.warning(
                            message
                        )

# ==========================================
# ATTENDANCE RECORDS
# ==========================================

elif menu == "Attendance Records":

    st.header("📊 Attendance Records")

    # Read attendance CSV
    df = pd.read_csv(
        ATTENDANCE_FILE
    )

    # Check records
    if df.empty:

        st.info(
            "No attendance records available."
        )

    else:

        st.success(
            f"Total records: {len(df)}"
        )

        # Display records
        st.dataframe(
            df,
            use_container_width=True
        )

        # Convert to CSV
        csv = df.to_csv(
            index=False
        ).encode("utf-8")

        # Download button
        st.download_button(
            label="📥 Download Attendance",
            data=csv,
            file_name="attendance.csv",
            mime="text/csv"
        )

Writing app.py


In [3]:
# MAKE ATTENDANCE FOLDER
import os
import pandas as pd

# Create attendance folder
os.makedirs("/content/attendance", exist_ok=True)

# Create attendance CSV if it doesn't exist
attendance_file = "/content/attendance/attendance.csv"

if not os.path.exists(attendance_file):

    df = pd.DataFrame(
        columns=[
            "Name",
            "Date",
            "Time",
            "Status"
        ]
    )

    df.to_csv(
        attendance_file,
        index=False
    )

print("Attendance folder and file are ready!")

Attendance folder and file are ready!


In [4]:
!ls -l /content/attendance/

total 4
-rw-r--r-- 1 root root 22 Aug 17 05:28 attendance.csv


In [5]:
# READ THE ATTENDANCE CSV
import pandas as pd

df = pd.read_csv("/content/attendance/attendance.csv")

print(df)

Empty DataFrame
Columns: [Name, Date, Time, Status]
Index: []


In [6]:
# START THE STREAMLIT APPLICATION
!streamlit run app.py &>/content/logs.txt &


In [7]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared



In [8]:
!chmod +x /content/cloudflared


In [9]:
!pkill -f streamlit
!pkill -f cloudflared
!nohup streamlit run /content/app.py --server.address 0.0.0.0 --server.port 8501 > /content/logs.txt 2>&1 &

In [10]:
import time
time.sleep(5)
print("Waiting completed")

Waiting completed


In [11]:
!cat /content/logs.txt



2026-08-17 05:29:03.465 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.231.236.81:8501



In [12]:
# SEND A REQUEST TO THE STREAMLIT SERVER
!curl -I http://localhost:8501

HTTP/1.1 200 OK
date: Mon, 17 Aug 2026 05:29:03 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 10951
last-modified: Mon, 17 Aug 2026 05:28:53 GMT
etag: "cde288b491fcc3736c9cb9f4a4d42853"
cache-control: no-cache



In [13]:
# CREATE A TEMPORARY TUNNEL
!nohup ./cloudflared tunnel --url http://localhost:8501 > /content/cloudflared.log 2>&1 &

In [14]:
import time
time.sleep(5)

In [15]:
# SEARCH FOR TEMPORARY URL AND GIVE THE URL
!grep -o "https://[-a-zA-Z0-9]*\.trycloudflare\.com" /content/cloudflared.log

https://thursday-repository-factor-specially.trycloudflare.com
